# Spark SQL via PySpark

Alternative to the spylon-kernel notebook. Runs Spark SQL through `spark.sql()`
using the PySpark kernel — handy if you prefer staying in Python.

In [1]:
import sys
sys.path.insert(0, '/workspace')
from scripts.spark_session import get_spark, JDBC_URL, JDBC_PROPS, BRONZE, GOLD

spark = get_spark('sql_via_pyspark')

def sql(query: str):
    """Run a Spark SQL query and display results."""
    return spark.sql(query).show(truncate=False)

print(f'Spark {spark.version} ready')

26/03/20 06:08:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/20 06:08:03 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/20 06:08:03 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


Spark 3.5.1 ready


In [2]:
spark.read.parquet(f'{BRONZE}/customers').createOrReplaceTempView('customers')
spark.read.parquet(f'{BRONZE}/orders').createOrReplaceTempView('orders')
print('Registered: customers, orders')

Registered: customers, orders


In [3]:
sql("""
    SELECT c.name, c.region,
           COUNT(o.order_id) AS num_orders,
           ROUND(SUM(o.quantity * o.unit_price), 2) AS total_spend
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    GROUP BY c.name, c.region
    ORDER BY total_spend DESC
""")

+-----------------+------+----------+-----------+
|name             |region|num_orders|total_spend|
+-----------------+------+----------+-----------+
|Northwind Traders|East  |2         |1199.65    |
|Contoso Ltd      |West  |3         |999.82     |
|Tailspin Toys    |East  |1         |899.70     |
|Adventure Works  |West  |2         |799.90     |
|Fabrikam Inc     |South |2         |679.87     |
+-----------------+------+----------+-----------+



In [4]:
spark.read.jdbc(JDBC_URL, 'dbo.orders', properties=JDBC_PROPS) \
    .createOrReplaceTempView('mssql_orders')

sql("""
    SELECT product, SUM(quantity) AS total_qty
    FROM mssql_orders
    GROUP BY product
    ORDER BY total_qty DESC
""")

+--------+---------+
|product |total_qty|
+--------+---------+
|Widget A|60       |
|Widget C|27       |
|Widget B|13       |
|Gadget X|3        |
|Gadget Y|3        |
+--------+---------+

